# 04 — Error Analysis

Deep-dive analyses:
1. Accuracy by sentence-length bucket
2. Per-dependency-relation accuracy
3. Long-distance arc stress-test (distance > 5)
4. Top confusion pairs

In [ ]:
import sys
sys.path.insert(0, "..")

from collections import Counter
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from src.data import load_sentences, bucket_by_length
from src.parsers import SpacyParser, StanzaParser
from src.metrics import uas, las, Gold, Prediction, _is_punct

SENTS = {
    "en": load_sentences(Path("../data/en_ewt_test.conllu")),
    "ru": load_sentences(Path("../data/ru_syntagrus_test.conllu")),
}
PARSERS = {
    "en": [SpacyParser("en_core_web_trf"), StanzaParser("en")],
    "ru": [SpacyParser("ru_core_news_lg"), StanzaParser("ru")],
}
print("Parsers loaded.")

In [ ]:
# Re-parse everything and keep predictions
CHUNK = 100
PARSES = {}  # (lang, parser_name) -> list[Prediction]

for lang, sents in SENTS.items():
    for parser in PARSERS[lang]:
        toks = [s.tokens for s in sents]
        results = []
        for i in tqdm(range(0, len(toks), CHUNK), desc=f"{lang}-{parser.name}"):
            results.extend(parser.parse(toks[i:i+CHUNK]))
        PARSES[(lang, parser.name)] = [Prediction(r.heads, r.deprels) for r in results]

print("Done parsing.")

In [ ]:
# 1. Accuracy by sentence length bucket
rows = []
for lang, sents in SENTS.items():
    buckets = bucket_by_length(sents)
    sent_to_idx = {id(s): i for i, s in enumerate(sents)}
    for bucket_name, bucket_sents in buckets.items():
        idxs = [sent_to_idx[id(s)] for s in bucket_sents]
        if not idxs:
            continue
        for (lang_, pname), preds in PARSES.items():
            if lang_ != lang:
                continue
            sub_pred = [preds[i] for i in idxs]
            sub_gold = [Gold(sents[i].heads, sents[i].deprels) for i in idxs]
            rows.append({
                "lang": lang, "parser": pname, "bucket": bucket_name,
                "n": len(idxs),
                "uas": round(uas(sub_pred, sub_gold), 4),
                "las": round(las(sub_pred, sub_gold), 4),
            })

df_len = pd.DataFrame(rows)
df_len.to_csv("../results/accuracy_by_length.csv", index=False)
print("Saved accuracy_by_length.csv")
df_len

In [ ]:
# 2. Per-relation accuracy
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    per_rel_total: Counter = Counter()
    per_rel_correct: Counter = Counter()
    for pred, sent in zip(preds, sents):
        for ph, pd_, gh, gd in zip(pred.heads, pred.deprels, sent.heads, sent.deprels):
            if _is_punct(gd):
                continue
            key = gd.split(":")[0]
            per_rel_total[key] += 1
            if ph == gh and pd_.split(":")[0] == key:
                per_rel_correct[key] += 1
    for rel, total in per_rel_total.items():
        rows.append({
            "lang": lang, "parser": pname, "relation": rel,
            "n": total,
            "accuracy": round(per_rel_correct[rel] / total, 4) if total else 0.0,
        })

df_rel = pd.DataFrame(rows).sort_values(["lang", "relation", "parser"])
df_rel.to_csv("../results/accuracy_by_relation.csv", index=False)
print("Saved accuracy_by_relation.csv")
df_rel.head(20)

In [ ]:
# 3. Long-distance arc stress-test (distance > 5)
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    lt = lc_uas = lc_las = 0
    for pred, sent in zip(preds, sents):
        for i, (ph, pd_, gh, gd) in enumerate(
            zip(pred.heads, pred.deprels, sent.heads, sent.deprels)
        ):
            if _is_punct(gd):
                continue
            dep_pos = i + 1
            dist = abs(dep_pos - gh) if gh != 0 else dep_pos
            if dist <= 5:
                continue
            lt += 1
            if ph == gh:
                lc_uas += 1
            if ph == gh and pd_.split(":")[0] == gd.split(":")[0]:
                lc_las += 1
    rows.append({
        "lang": lang, "parser": pname,
        "n_long_arcs": lt,
        "uas_long": round(lc_uas / lt, 4) if lt else 0.0,
        "las_long": round(lc_las / lt, 4) if lt else 0.0,
    })

df_long = pd.DataFrame(rows)
df_long.to_csv("../results/long_distance_stress.csv", index=False)
print("Saved long_distance_stress.csv")
df_long

In [ ]:
# 4. Top confusion pairs (gold label → predicted label)
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    conf: Counter = Counter()
    for pred, sent in zip(preds, sents):
        for pd_, gd in zip(pred.deprels, sent.deprels):
            if _is_punct(gd):
                continue
            p_label = pd_.split(":")[0]
            g_label = gd.split(":")[0]
            if p_label != g_label:
                conf[(g_label, p_label)] += 1
    for (g, p), n in conf.most_common(20):
        rows.append({"lang": lang, "parser": pname, "gold": g, "predicted": p, "count": n})

df_conf = pd.DataFrame(rows)
df_conf.to_csv("../results/confusion_top.csv", index=False)
print("Saved confusion_top.csv")
df_conf.head(20)

## What we learned
- Long-distance arcs reveal where graph-based wins (global optimization helps)
- Russian shows larger gap than English (freer word order = more non-projective arcs)
- Per-relation breakdown shows which syntactic relations are hardest
- Ready for poster figures (notebook 05)